# Proyecto Final ML — Clasificación de Calidad de Café (CQI)
**Curso:** Aplicación de Modelos de Machine Learning  
**Dataset:** Coffee Quality Institute (CQI) — Arabica  
**Objetivo:** Clasificar muestras de café como *Specialty* (≥80 puntos) o *No Specialty* (<80 puntos)

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                             roc_curve, ConfusionMatrixDisplay, f1_score, accuracy_score)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
import xgboost as xgb

# Estilo general
plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.family'] = 'DejaVu Sans'
PALETTE = ['#4e8c3f', '#c0392b']  # Verde café / Rojo
sns.set_style("whitegrid")

3.2.0


## 1. Carga y Exploración de Datos (EDA)

In [6]:
# =============================================================================
print("=" * 60)
print("1. CARGA Y EXPLORACIÓN DE DATOS")
print("=" * 60)

df = pd.read_csv('coffee_quality.csv')

print(f"\nDimensiones: {df.shape[0]} filas × {df.shape[1]} columnas")
print(f"\nPrimeras filas:")
print(df.head(3).to_string())
print(f"\nTipos de datos y nulos:")
info = pd.DataFrame({
    'Tipo': df.dtypes,
    'Nulos': df.isnull().sum(),
    '% Nulos': (df.isnull().sum() / len(df) * 100).round(2)
})
print(info[info['Nulos'] > 0])
print(f"\nEstadísticas descriptivas (variables numéricas):")
print(df.describe().round(2).to_string())

# Variable objetivo: Specialty vs No Specialty
df['Quality'] = (df['Total_Cup_Points'] >= 80).astype(int)
df['Quality_Label'] = df['Quality'].map({1: 'Specialty (≥80)', 0: 'No Specialty (<80)'})

print(f"\nDistribución de clases:")
print(df['Quality_Label'].value_counts())
print(f"Proporción: {df['Quality'].mean():.1%} specialty")

# ---------- FIGURA 1: Distribución del puntaje y clases ----------
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Exploración de Calidad del Café (CQI)', fontsize=14, fontweight='bold', y=1.02)

# Histograma de puntaje total
ax = axes[0]
ax.hist(df[df['Quality'] == 0]['Total_Cup_Points'], bins=20, color=PALETTE[1],
        alpha=0.7, label='No Specialty', edgecolor='white')
ax.hist(df[df['Quality'] == 1]['Total_Cup_Points'], bins=20, color=PALETTE[0],
        alpha=0.7, label='Specialty', edgecolor='white')
ax.axvline(80, color='black', linestyle='--', lw=1.5, label='Umbral = 80')
ax.set_xlabel('Total Cup Points')
ax.set_ylabel('Frecuencia')
ax.set_title('Distribución del Puntaje Total')
ax.legend()

# Pie chart de clases
ax = axes[1]
counts = df['Quality_Label'].value_counts()
ax.pie(counts, labels=counts.index, colors=PALETTE[::-1],
       autopct='%1.1f%%', startangle=90,
       wedgeprops={'edgecolor': 'white', 'linewidth': 2})
ax.set_title('Distribución de Clases')

# Puntaje promedio por país
ax = axes[2]
country_avg = (df.groupby('Country_of_Origin')['Total_Cup_Points']
               .mean().sort_values(ascending=True).tail(10))
bars = ax.barh(country_avg.index, country_avg.values, color='#4e8c3f', alpha=0.8, edgecolor='white')
ax.set_xlabel('Puntaje Promedio')
ax.set_title('Top 10 Países por Puntaje Promedio')
ax.set_xlim(country_avg.min() - 0.3, country_avg.max() + 0.3)
for bar, val in zip(bars, country_avg.values):
    ax.text(val + 0.02, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('fig1_distribucion.png', bbox_inches='tight')
plt.close()
print("\n✓ Figura 1 guardada: Distribución y clases")

# ---------- FIGURA 2: Correlaciones y atributos sensoriales ----------
sensory_cols = ['Aroma', 'Flavor', 'Aftertaste', 'Acidity', 'Body',
                'Balance', 'Uniformity', 'Clean_Cup', 'Sweetness',
                'Cupper_Points', 'Total_Cup_Points']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Análisis de Atributos Sensoriales', fontsize=14, fontweight='bold')

# Mapa de calor de correlaciones
corr = df[sensory_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='YlGn',
            ax=axes[0], linewidths=0.5, vmin=0, vmax=1,
            annot_kws={'size': 7})
axes[0].set_title('Correlación entre Atributos Sensoriales')
axes[0].tick_params(axis='x', rotation=45)

# Boxplot de atributos por clase
attrs = ['Aroma', 'Flavor', 'Aftertaste', 'Acidity', 'Body', 'Balance']
df_melt = df[attrs + ['Quality_Label']].melt(id_vars='Quality_Label',
                                              var_name='Atributo', value_name='Puntuación')
sns.boxplot(data=df_melt, x='Atributo', y='Puntuación', hue='Quality_Label',
            palette=PALETTE[::-1], ax=axes[1], width=0.6, linewidth=0.8)
axes[1].set_title('Atributos Sensoriales por Clase')
axes[1].legend(title='Categoría', loc='lower right')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('fig2_correlaciones.png', bbox_inches='tight')
plt.close()
print("✓ Figura 2 guardada: Correlaciones y atributos")

# ---------- FIGURA 3: Variables categóricas ----------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Análisis de Variables Categóricas', fontsize=14, fontweight='bold')

# Tasa de specialty por método de procesamiento
proc_rate = df.groupby('Processing_Method')['Quality'].mean().sort_values(ascending=False)
bars = axes[0].bar(proc_rate.index, proc_rate.values * 100,
                   color='#4e8c3f', alpha=0.8, edgecolor='white')
axes[0].set_ylabel('% Specialty')
axes[0].set_title('Tasa de Specialty por Método de Procesamiento')
axes[0].set_ylim(0, 105)
axes[0].tick_params(axis='x', rotation=20)
for bar, val in zip(bars, proc_rate.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val*100:.1f}%', ha='center', fontsize=9)

# Altitud vs puntaje
axes[1].scatter(df[df['Quality']==0]['Altitude_mean_meters'],
                df[df['Quality']==0]['Total_Cup_Points'],
                c=PALETTE[1], alpha=0.4, s=15, label='No Specialty')
axes[1].scatter(df[df['Quality']==1]['Altitude_mean_meters'],
                df[df['Quality']==1]['Total_Cup_Points'],
                c=PALETTE[0], alpha=0.3, s=15, label='Specialty')
axes[1].set_xlabel('Altitud (metros)')
axes[1].set_ylabel('Total Cup Points')
axes[1].set_title('Altitud vs Puntaje de Calidad')
axes[1].axhline(80, color='black', linestyle='--', lw=1, alpha=0.5)
axes[1].legend()

plt.tight_layout()
plt.savefig('fig3_categoricas.png', bbox_inches='tight')
plt.close()
print("✓ Figura 3 guardada: Variables categóricas")

1. CARGA Y EXPLORACIÓN DE DATOS

Dimensiones: 1200 filas × 20 columnas

Primeras filas:
   Species Country_of_Origin   Variety Processing_Method         Color  Altitude_mean_meters  Aroma  Flavor  Aftertaste  Acidity  Body  Balance  Uniformity  Clean_Cup  Sweetness  Cupper_Points  Total_Cup_Points  Moisture  Category_One_Defects  Category_Two_Defects
0  Arabica          Colombia  Castillo      Washed / Wet         Green                1365.0   7.80    7.55        7.11     7.17  7.68     7.05        9.74       9.85       9.36           8.07             81.81     11.21                     0                     1
1  Arabica            Brazil    Typica      Washed / Wet  Bluish-Green                1836.0   7.54    7.33        7.47     7.21  7.48     7.25        9.72      10.00      10.00           7.35             81.40      9.86                     0                     1
2  Arabica             Kenya   Bourbon      Washed / Wet         Green                2065.0   7.86    7.55        7.

## 2. Preprocesamiento

In [7]:
# =============================================================================
print("\n" + "=" * 60)
print("2. PREPROCESAMIENTO")
print("=" * 60)

# Features y target
feature_cols = ['Aroma', 'Flavor', 'Aftertaste', 'Acidity', 'Body', 'Balance',
                'Uniformity', 'Clean_Cup', 'Sweetness', 'Cupper_Points',
                'Moisture', 'Category_One_Defects', 'Category_Two_Defects',
                'Altitude_mean_meters', 'Processing_Method', 'Color', 'Variety']

X = df[feature_cols].copy()
y = df['Quality']

# Codificación de variables categóricas
cat_cols = ['Processing_Method', 'Color', 'Variety']
num_cols = [c for c in feature_cols if c not in cat_cols]

# Imputación: numérico → mediana, categórico → moda
for col in num_cols:
    X[col] = X[col].fillna(X[col].median())
for col in cat_cols:
    X[col] = X[col].fillna(X[col].mode()[0])

# Label encoding para categóricas
le = LabelEncoder()
for col in cat_cols:
    X[col] = le.fit_transform(X[col])

print(f"Features usados: {len(feature_cols)}")
print(f"Nulos restantes: {X.isnull().sum().sum()}")

# División train / test (80/20, estratificada)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nConjunto de entrenamiento: {X_train.shape[0]} muestras")
print(f"Conjunto de prueba:        {X_test.shape[0]} muestras")

# Escalado (solo para Logistic Regression)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)


2. PREPROCESAMIENTO
Features usados: 17
Nulos restantes: 0

Conjunto de entrenamiento: 960 muestras
Conjunto de prueba:        240 muestras


## 3. Entrenamiento de Modelos

In [8]:
# =============================================================================
print("\n" + "=" * 60)
print("3. ENTRENAMIENTO Y EVALUACIÓN DE MODELOS")
print("=" * 60)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# --- Modelo 1: Regresión Logística (baseline) ---
lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr_cv = cross_val_score(lr, X_train_sc, y_train, cv=cv, scoring='f1')
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)
y_prob_lr = lr.predict_proba(X_test_sc)[:, 1]
print(f"\n[1] Regresión Logística")
print(f"    CV F1 (mean ± std): {lr_cv.mean():.4f} ± {lr_cv.std():.4f}")
print(f"    Test Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"    Test F1:       {f1_score(y_test, y_pred_lr):.4f}")
print(f"    Test AUC-ROC:  {roc_auc_score(y_test, y_prob_lr):.4f}")

# --- Modelo 2: Random Forest ---
rf = RandomForestClassifier(n_estimators=200, random_state=42,
                            class_weight='balanced', n_jobs=-1)
rf_cv = cross_val_score(rf, X_train, y_train, cv=cv, scoring='f1')
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]
print(f"\n[2] Random Forest")
print(f"    CV F1 (mean ± std): {rf_cv.mean():.4f} ± {rf_cv.std():.4f}")
print(f"    Test Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"    Test F1:       {f1_score(y_test, y_pred_rf):.4f}")
print(f"    Test AUC-ROC:  {roc_auc_score(y_test, y_prob_rf):.4f}")

# --- Modelo 3: XGBoost ---
xgb_model = xgb.XGBClassifier(
    n_estimators=200, learning_rate=0.05, max_depth=4,
    use_label_encoder=False, eval_metric='logloss',
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    random_state=42, n_jobs=-1
)
xgb_cv = cross_val_score(xgb_model, X_train, y_train, cv=cv, scoring='f1')
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]
print(f"\n[3] XGBoost")
print(f"    CV F1 (mean ± std): {xgb_cv.mean():.4f} ± {xgb_cv.std():.4f}")
print(f"    Test Accuracy: {accuracy_score(y_test, y_pred_xgb):.4f}")
print(f"    Test F1:       {f1_score(y_test, y_pred_xgb):.4f}")
print(f"    Test AUC-ROC:  {roc_auc_score(y_test, y_prob_xgb):.4f}")


3. ENTRENAMIENTO Y EVALUACIÓN DE MODELOS

[1] Regresión Logística
    CV F1 (mean ± std): 0.9641 ± 0.0123
    Test Accuracy: 0.9208
    Test F1:       0.9542
    Test AUC-ROC:  0.9888

[2] Random Forest
    CV F1 (mean ± std): 0.9514 ± 0.0032
    Test Accuracy: 0.9125
    Test F1:       0.9536
    Test AUC-ROC:  0.9657

[3] XGBoost
    CV F1 (mean ± std): 0.9489 ± 0.0131
    Test Accuracy: 0.9125
    Test F1:       0.9508
    Test AUC-ROC:  0.9221


## 4. Optimización de Hiperparámetros

In [9]:
# =============================================================================
print("\n" + "=" * 60)
print("4. OPTIMIZACIÓN DE HIPERPARÁMETROS (Random Forest)")
print("=" * 60)

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'max_features': ['sqrt', 'log2']
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42, class_weight='balanced', n_jobs=-1),
    param_grid, cv=cv, scoring='f1', n_jobs=-1, verbose=0
)
grid_search.fit(X_train, y_train)

best_rf = grid_search.best_estimator_
y_pred_best = best_rf.predict(X_test)
y_prob_best = best_rf.predict_proba(X_test)[:, 1]

print(f"Mejores hiperparámetros: {grid_search.best_params_}")
print(f"Mejor CV F1: {grid_search.best_score_:.4f}")
print(f"Test Accuracy: {accuracy_score(y_test, y_pred_best):.4f}")
print(f"Test F1:       {f1_score(y_test, y_pred_best):.4f}")
print(f"Test AUC-ROC:  {roc_auc_score(y_test, y_prob_best):.4f}")
print("\nReporte de clasificación (RF optimizado):")
print(classification_report(y_test, y_pred_best,
                             target_names=['No Specialty', 'Specialty']))


4. OPTIMIZACIÓN DE HIPERPARÁMETROS (Random Forest)
Mejores hiperparámetros: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_split': 5, 'n_estimators': 100}
Mejor CV F1: 0.9551
Test Accuracy: 0.9250
Test F1:       0.9600
Test AUC-ROC:  0.9597

Reporte de clasificación (RF optimizado):
              precision    recall  f1-score   support

No Specialty       1.00      0.25      0.40        24
   Specialty       0.92      1.00      0.96       216

    accuracy                           0.93       240
   macro avg       0.96      0.62      0.68       240
weighted avg       0.93      0.93      0.90       240



## 5. Visualización de Resultados

In [12]:
# =============================================================================
print("\n" + "=" * 60)
print("5. VISUALIZACIÓN DE RESULTADOS")
print("=" * 60)

models_info = {
    'Reg. Logística': (y_pred_lr, y_prob_lr),
    'Random Forest':  (y_pred_rf, y_prob_rf),
    'XGBoost':        (y_pred_xgb, y_prob_xgb),
    'RF Optimizado':  (y_pred_best, y_prob_best),
}

# ---------- FIGURA 4: Comparación de métricas ----------
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Comparación de Modelos', fontsize=14, fontweight='bold')

names = list(models_info.keys())
accs  = [accuracy_score(y_test, p) for p, _ in models_info.values()]
f1s   = [f1_score(y_test, p) for p, _ in models_info.values()]
aucs  = [roc_auc_score(y_test, pb) for _, pb in models_info.values()]

colors_bar = ['#a8d5a2', '#4e8c3f', '#2d6a27', '#1a4018']

for ax, metric, values, title in zip(
        axes,
        [accs, f1s, aucs],
        [accs, f1s, aucs],
        ['Accuracy', 'F1-Score', 'AUC-ROC']):
    bars = ax.bar(names, values, color=colors_bar, edgecolor='white', width=0.6)
    ax.set_ylim(min(values) - 0.05, 1.02)
    ax.set_title(title, fontweight='bold')
    ax.tick_params(axis='x', rotation=15)
    ax.set_ylabel('Valor')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.4f}', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('fig4_comparacion.png', bbox_inches='tight')
plt.close()
print("✓ Figura 4 guardada: Comparación de modelos")

# ---------- FIGURA 5: Curvas ROC + Matrices de confusión ----------
fig = plt.figure(figsize=(16, 6))
gs = gridspec.GridSpec(1, 5, figure=fig)
fig.suptitle('Curvas ROC y Matriz de Confusión (RF Optimizado)', fontsize=13, fontweight='bold')

# Curvas ROC
ax_roc = fig.add_subplot(gs[0, :3])
plot_colors = ['#999999', '#4e8c3f', '#c0392b', '#1a4018']
for (name, (_, prob)), col in zip(models_info.items(), plot_colors):
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc_val = roc_auc_score(y_test, prob)
    ax_roc.plot(fpr, tpr, label=f'{name} (AUC={auc_val:.3f})', color=col, lw=2)
ax_roc.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
ax_roc.set_xlabel('Tasa de Falsos Positivos')
ax_roc.set_ylabel('Tasa de Verdaderos Positivos')
ax_roc.set_title('Curvas ROC')
ax_roc.legend(loc='lower right')

# Matriz de confusión RF optimizado
ax_cm = fig.add_subplot(gs[0, 3:])
cm = confusion_matrix(y_test, y_pred_best)
disp = ConfusionMatrixDisplay(cm, display_labels=['No Specialty', 'Specialty'])
disp.plot(ax=ax_cm, colorbar=False, cmap='Greens')
ax_cm.set_title('Matriz de Confusión\n(RF Optimizado)')

plt.tight_layout()
plt.savefig('fig5_roc_cm.png', bbox_inches='tight')
plt.close()
print("✓ Figura 5 guardada: ROC y matrices de confusión")

# ---------- FIGURA 6: Importancia de variables ----------
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Importancia de Variables', fontsize=14, fontweight='bold')

# Random Forest feature importance
importances = pd.Series(best_rf.feature_importances_, index=feature_cols).sort_values()
colors_imp = ['#c0392b' if i >= len(importances) - 3 else '#4e8c3f'
              for i in range(len(importances))]
importances.plot(kind='barh', ax=axes[0], color=colors_imp, edgecolor='white')
axes[0].set_title('RF: Importancia de Variables (Gini)')
axes[0].set_xlabel('Importancia')
axes[0].axvline(importances.mean(), color='black', linestyle='--',
                lw=1, alpha=0.6, label='Promedio')
axes[0].legend()

# XGBoost feature importance
xgb_imp = pd.Series(xgb_model.feature_importances_, index=feature_cols).sort_values()
colors_xgb = ['#c0392b' if i >= len(xgb_imp) - 3 else '#4e8c3f'
               for i in range(len(xgb_imp))]
xgb_imp.plot(kind='barh', ax=axes[1], color=colors_xgb, edgecolor='white')
axes[1].set_title('XGBoost: Importancia de Variables')
axes[1].set_xlabel('Importancia')
axes[1].axvline(xgb_imp.mean(), color='black', linestyle='--',
                lw=1, alpha=0.6, label='Promedio')
axes[1].legend()

plt.tight_layout()
plt.savefig('fig6_importancia.png', bbox_inches='tight')
plt.close()
print("✓ Figura 6 guardada: Importancia de variables")


5. VISUALIZACIÓN DE RESULTADOS
✓ Figura 4 guardada: Comparación de modelos
✓ Figura 5 guardada: ROC y matrices de confusión
✓ Figura 6 guardada: Importancia de variables


## 6. Resumen Final

In [14]:
# =============================================================================
print("\n" + "=" * 60)
print("6. RESUMEN FINAL DE RESULTADOS")
print("=" * 60)

results_df = pd.DataFrame({
    'Modelo': names,
    'Accuracy': [f'{v:.4f}' for v in accs],
    'F1-Score': [f'{v:.4f}' for v in f1s],
    'AUC-ROC':  [f'{v:.4f}' for v in aucs],
})
print(results_df.to_string(index=False))

top3 = pd.Series(best_rf.feature_importances_, index=feature_cols).nlargest(3)
print(f"\nTop 3 variables más importantes (RF Optimizado):")
for feat, val in top3.items():
    print(f"  {feat}: {val:.4f}")

print("\nPipeline completo ejecutado exitosamente.")


6. RESUMEN FINAL DE RESULTADOS
        Modelo Accuracy F1-Score AUC-ROC
Reg. Logística   0.9208   0.9542  0.9888
 Random Forest   0.9125   0.9536  0.9657
       XGBoost   0.9125   0.9508  0.9221
 RF Optimizado   0.9250   0.9600  0.9597

Top 3 variables más importantes (RF Optimizado):
  Cupper_Points: 0.1509
  Acidity: 0.1159
  Body: 0.1155

Pipeline completo ejecutado exitosamente.
